In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Row

In [0]:
catalog_name = "ecommerce"

### Products

In [0]:
df_products = spark.table("ecommerce.silver.slv_products")
df_category = spark.table("ecommerce.silver.slv_category")
df_brands = spark.table("ecommerce.silver.slv_brands")

In [0]:
df_products.createOrReplaceTempView("v_products")
df_category.createOrReplaceTempView("v_category")
df_brands.createOrReplaceTempView("v_brands")

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
%sql
CREATE OR REPLACE TABLE gold.gld_dim_products AS
SELECT 
    p.product_id,
    p.sku,
    c.category_name,
    b.brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p.file_name,
    p.ingest_timestamp
FROM v_products p
LEFT JOIN v_category c
ON p.category_code = c.category_code
LEFT JOIN v_brands b
ON p.brand_code = b.brand_code
LIMIT 5

### Customers

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  


In [0]:
rows = []

for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))

rows[:10]

In [0]:
df_region_mapping = spark.createDataFrame(rows)

df_region_mapping.show(10)

In [0]:
df_silver = spark.table("ecommerce.silver.slv_customers")
df_silver.show(10)

In [0]:
df_silver.createOrReplaceTempView("slv_customers")
df_region_mapping.createOrReplaceTempView("v_region_mapping")

In [0]:
df_silver.printSchema()

In [0]:
%sql
SELECT
    sc.customer_id,
    sc.phone,
    sc.country_code,
    sc.country,
    sc.state,
    rm.region,
    sc.file_name,
    sc.ingest_timestamp
FROM slv_customers sc
LEFT JOIN v_region_mapping rm
ON sc.country = rm.country AND sc.state = rm.state
LIMIT 10

In [0]:
df_gold = _sqldf
df_gold.show(10)

In [0]:
df_gold.select("region").filter(col("region").isNull()).count()

In [0]:
df_gold.write.format("delta")\
             .mode("overwrite")\
             .option("mergeSchema", True)\
             .saveAsTable(f"{catalog_name}.gold.gld_customers")

### Date/Calendar

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Row

catalog_name = "ecommerce"

In [0]:
df_silver = spark.table(f"{catalog_name}.silver.slv_calendar")
df_silver.show(10)

In [0]:
df_gold = df_silver.select(date_format(col("date"), "yyyyMMdd").alias("date_id"), "*")
df_gold.show(10)

In [0]:
df_gold = df_gold.select("date_id", "date", "year", "quarter", date_format(col("date"), "MMMM").alias("month"), "week", "day_name", "_ingested_at", "_source_file")
df_gold.show(10)

In [0]:
df_gold = df_gold.withColumn("is_weekend", when(col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0))
df_gold.show(10)

In [0]:
desired_columns_order = ["date_id", "date", "year", "month", "day_name", "is_weekend", "quarter", "week", "_ingested_at", "_source_file"]

df_gold = df_gold.select(desired_columns_order)

display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta")\
             .mode("overwrite")\
             .option("mergeSchema", True)\
             .saveAsTable(f"{catalog_name}.gold.gld_dim_date")

In [0]:
%sql
ALTER TABLE ecommerce.gold.gld_customers RENAME TO ecommerce.gold.gld_dim_customers;